# tqdm-postfix-metrics — worked example 2: Postfix shows current learning rate and step loss

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `tqdm-postfix-metrics`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

When optimizers change their learning rate (e.g. via a scheduler), surfacing it in the tqdm postfix helps you see the schedule unfold during training. The pattern is identical to tracking loss: accumulate into a sidecar list and call `pbar.set_postfix(loss=..., lr=...)` on each step. The progress bar updates in-place each iteration so only the latest values are visible.

## Worked solution

**Step 1 – Build a simple 1-param model and optimizer.** We create a scalar weight and an SGD optimizer. We also build a `LambdaLR` scheduler that halves the LR every two steps to make the LR change visible.

**Step 2 – Wrap the step range in tqdm.** `pbar = tqdm(range(n_steps), desc='Training')` gives us a bar over the step indices. No `enumerate` needed here since we just need the step count.

**Step 3 – Per-step training cycle.** Standard forward-loss-backward-step-zero_grad cycle. After `optimizer.step()` we read the current LR from `optimizer.param_groups[0]['lr']` and call `pbar.set_postfix(loss=f'{loss:.4f}', lr=f'{current_lr:.5f}')`.

**Step 4 – Scheduler step.** `scheduler.step()` is called AFTER the optimizer step so the LR update is visible on the NEXT postfix call.

**Step 5 – Return.** We return the sidecar log so the test can verify each entry has both keys and the LR decreased as expected.

In [ ]:
from tqdm import tqdm
import torch as t

def worked2_tqdm_loss_lr(n_steps=6):
    """
    Training loop showing live loss + lr in tqdm postfix.
    Returns list of dicts recorded at each step.
    """
    t.manual_seed(7)
    w = t.tensor([3.0], requires_grad=True)
    x = t.linspace(0, 1, 10)
    y_target = 2.0 * x
    optimizer = t.optim.SGD([w], lr=0.1)
    scheduler = t.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda step: 0.7 ** step)

    log = []
    pbar = tqdm(range(n_steps), desc='Training')
    for step in pbar:
        pred = w * x
        loss = ((pred - y_target) ** 2).mean()
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        current_lr = optimizer.param_groups[0]['lr']
        pf = dict(loss=f'{loss.item():.4f}', lr=f'{current_lr:.5f}')
        pbar.set_postfix(**pf)
        log.append(pf)
        scheduler.step()
    return log

log = worked2_tqdm_loss_lr()
print('step 0:', log[0])
print('step 5:', log[5])